# LA-ef: build, prepare, run and plot

One notebook for the whole project. Everything below runs from the repository root and uses the
helper package in `laef_nb/`. The helpers hold the plumbing so that this notebook stays a sequence
of decisions rather than a pile of shell commands.

Order of work:

1. settings
2. environment check
3. build the C++
4. dataset artifacts
5. index, calibration and one profile per thread count
6. the thread-scaling study
7. results, diagnostics and figures

A step that already has its output on disk is skipped, so the notebook can be rerun safely and an
interrupted study continues where it stopped.

## 1. Settings

This is the only cell you normally edit.

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import laef_nb as nb

pd.set_option("display.width", 200, "display.max_columns", 60)
nb.figures.use_style()

STUDY = nb.Study(
    name="thread_scaling",
    datasets=("synthetic-small",),
    threads=(2, 4, 8, 16, 24),
    la_policies=("laef_uniform", "laef_wf"),
    rhos=(0.6, 0.8, 1.0, 1.2, 1.3),
    slo_fractions=(0.9,),
    duration_s=60.0,
    repeats=1,
    arrival="constant",     # constant or poisson. step, mmpp2 and trace were removed from the engine.
)

SAT_SECONDS = 120.0      # saturation profile length per dataset and thread count
BUILD_NATIVE = True      # -march=native
INTERVAL_S = 1.0         # time bin for intervals.csv
KEEP_QUERIES = False     # keep the per-query csv of every run, which is large

print(f"{STUDY.n_runs} bench runs, about {STUDY.bench_hours:.1f} h of measurement")
print(f"results -> {STUDY.dir}")
print(f"experiment file -> {STUDY.yaml_path}")

## 2. Environment check

The thread sweep is only meaningful when every requested worker count fits on real cores. A row
marked `block` means the measured `mu_sat` at that thread count is a queueing artefact rather than a
capacity measurement, so remove it from `STUDY.threads` or run on a larger machine.

In [ ]:
INFO = nb.system.cpu_info()
nb.system.show(INFO)
print()
display(nb.system.check_threads(STUDY.threads, INFO, pin_cores=STUDY.pin_cores))
for w in nb.system.environment_warnings(INFO):
    print("warning:", w)

In [ ]:
# Keep only the thread counts this machine can actually run. Comment out to force the full sweep.
usable = nb.system.usable_threads(STUDY.threads, INFO)
if list(usable) != list(STUDY.threads):
    print(f"dropping {sorted(set(STUDY.threads) - set(usable))}: more workers than logical CPUs")
    STUDY = STUDY.with_(threads=tuple(usable))
print("threads:", STUDY.threads)

## 3. Build the C++

`ensure_adaef` clones `hnsw-ada-ef` at the pinned commit. Eigen and Boost.Math are taken from the
system when present and otherwise downloaded as headers into `cpp/third_party/vendor/`.

The engine tests are timing sensitive, so they run one at a time. Running them in parallel on a
small machine produces failures that are contention, not defects.

In [ ]:
nb.build.ensure_adaef()
nb.build.configure(native=BUILD_NATIVE)
nb.build.compile_()
print(nb.build.binaries())

In [ ]:
rc = nb.build.ctest(fast=True, jobs=1)
print("ctest exit", rc)

## 4. Dataset artifacts

Each dataset needs `train.npy`, `test.npy`, exact ground truth, calibration samplings and estimator
statistics. The ground truth and statistics steps are GPU work and normally run on Colab through
`gpu_preprocessing/colab_preprocess.ipynb`. For a small dataset the same steps run on CPU here,
which is what makes a self-contained smoke test possible.

In [ ]:
display(nb.datasets.catalogue())
display(nb.datasets.status(STUDY.datasets, STUDY.threads))

In [ ]:
# Path A, small dataset, everything local on CPU.
for ds in STUDY.datasets:
    if not (nb.ARTIFACTS / ds / "stats" / "estimator.bin").exists():
        nb.datasets.prepare_local(ds, device="cpu")

# Path B, a dataset preprocessed on Colab. Point this at the folder Colab wrote and rerun.
# nb.datasets.import_colab("glove-100-angular", "/mnt/d/laef_artifacts/glove-100-angular")

display(nb.datasets.status(STUDY.datasets, STUDY.threads))

## 5. Index, calibration and profiles

The HNSW index and the Ada-ef calibration tables do not depend on the worker count, so they are
built once per dataset. The profile does depend on it. Every thread count needs its own
`profile_C<threads>.json` because `mu_sat`, the unit cost and the saturation p95 that defines
`T_SLO` are all measured at that worker count.

This is also the first real result of the project, since `mu_sat` against C is the CPU scalability
curve the proposal asks for.

In [ ]:
PROFILES = nb.pipeline.prepare(STUDY.datasets, STUDY.threads, sat_seconds=SAT_SECONDS)
display(PROFILES)

In [ ]:
fig, _ = nb.figures.capacity_scaling(PROFILES, STUDY)
fig

The probe menu decides how much shorter-probe headroom the allocator has to work with. When only the
full probe survives calibration the allocator can move effort along the ef ladder but not along the
probe ladder, which limits how much work it can save.

In [ ]:
for ds in STUDY.datasets:
    display(nb.pipeline.probe_menu(ds))

## 6. The thread-scaling study

The generated experiment file holds one pair of sections per thread count, so results are written to
`results/<study>/c<threads>/ada/` and `results/<study>/c<threads>/laef/`.

Load is expressed as rho, a multiple of that thread count's own measured `mu_sat`. Every worker
count is therefore driven to the same relative pressure, which is what makes the comparison across C
meaningful. The absolute arrival rate differs between thread counts by design.

In [ ]:
print(nb.experiments.write_yaml(STUDY).read_text()[:1200])

`pending` counts what is finished, what started and died, and what is stale. A run is stale when its
`run_config.json` no longer matches the Study. The run id does not contain `duration_s`, so a
finished 10 s run is reused unchanged after you raise the duration to 120 s. Pass `force=True` to
`launch` to rerun those instead of mixing run lengths in one frame.

In [ ]:
problems = nb.experiments.check_ready(STUDY)
print("\n".join(problems) if problems else "all inputs present")
display(nb.experiments.pending(STUDY))
display(nb.experiments.stale(STUDY))

In [ ]:
# Dry run first. Remove the comment on the second line to measure.
nb.experiments.launch(STUDY, dry_run=True)
# nb.experiments.launch(STUDY, interval=INTERVAL_S, keep_queries=KEEP_QUERIES, force=False)

## 7. Results

`summary` is one row per run. `paired` puts every LA-ef run next to the stock Ada-ef run at the same
dataset, thread count, load and repeat. `allocator_delta` is the proportional cut minus the naive cut
at matched settings, which is the headline question of the project.

In [ ]:
DF = nb.collect.summary(STUDY)
PAIR = nb.collect.paired(DF)
DELTA = nb.collect.allocator_delta(DF)
print(DF.shape, PAIR.shape, DELTA.shape)
display(DF[["dataset", "threads", "policy", "rho", "slo_ms", "p95_ms", "slo_attainment",
            "mean_recall", "p5_recall", "mean_ef_ratio", "achieved_qps"]].head(20))

### Is the allocator actually cutting

If `mean_ef_ratio` sits near 1 then LA-ef is barely reducing effort, and any difference between the
proportional cut and the naive cut is noise rather than a result. Check this before reading anything
into the recall comparison. `forced_floor_rate` and `floor_mode_query_rate` show whether the floor
path is doing the work instead of the allocator.

In [ ]:
display(nb.collect.cut_diagnostics(DF))

In [ ]:
display(PAIR[["dataset", "threads", "policy", "rho", "p95_over_slo", "meets_slo",
              "recall_loss_pts", "p5_recall_loss_pts", "goodput_gain"]])

In [ ]:
display(DELTA)

## 8. Figures

Everything is written to `results/<study>/figures/`. When the Study has more than one SLO fraction
the figures show one of them rather than averaging across deadlines, and they say which one in the
title. Set `SHOW_FRACTION` to pick a different one.

In [ ]:
SHOW_FRACTION = None   # None picks the largest fraction present
IV = nb.collect.intervals(STUDY)
TRACE = DF[DF.policy == "laef_wf"].run_id.iloc[-1] if not DF.empty else None
paths = nb.figures.all_figures(PROFILES, DF, PAIR, DELTA, STUDY, IV, TRACE,
                               slo_fraction=SHOW_FRACTION)
for p in paths:
    print(p)

In [ ]:
from IPython.display import Image, display as show
for p in paths:
    show(Image(filename=str(p)))

## 9. Cross-check

The C++ writes `summary.json` itself. This recomputes the same metrics in pandas straight from
`queries.csv` and reports any metric that disagrees by more than the tolerance. It needs a study
launched with `keep_queries=True`, because the per-query file is deleted once `intervals.csv` exists.
Run it once after a change to the engine or the metric definitions.

In [ ]:
CHECK = nb.crosscheck.check_study(STUDY, tol=0.02)
if CHECK.empty:
    print("no queries.csv on disk. Relaunch with keep_queries=True to cross-check the engine summary.")
else:
    bad = CHECK[~CHECK.ok]
    print(f"{len(CHECK)} comparisons over {CHECK.run_id.nunique()} runs, {len(bad)} outside tolerance")
    display(bad if not bad.empty else CHECK.groupby("metric").rel_diff.max().to_frame("max_rel_diff"))

## 10. Export

`summary_tidy.csv` and `comparison_tidy.csv` sit next to the generated `summary_all.csv` and
`comparison.csv`. The tidy files carry the derived columns used by the figures.

In [ ]:
print(nb.collect.export(STUDY, DF, PAIR))

## 11. Other studies

The same machinery runs any grid. Change the fields and rerun from section 6.

An SLO sweep at one thread count, which is the proposal's question about how much recall must be
given up at each level of overload.

In [ ]:
SLO_STUDY = STUDY.with_(name="slo_fractions", threads=(8,),
                        slo_fractions=(0.95, 0.9, 0.8, 0.75),
                        rhos=(0.6, 0.8, 1.0, 1.2, 1.3))
print(SLO_STUDY.n_runs, "runs,", f"{SLO_STUDY.bench_hours:.1f} h")
# nb.experiments.launch(SLO_STUDY, interval=INTERVAL_S)

The arrival process. The engine supports `constant` and `poisson` only. The step, MMPP-2 and trace
profiles were removed, so the proposal's bursty workload requirement currently has no engine support.
Poisson arrivals give clustering in time and are the closest available stand-in, because the same
mean rate arrives unevenly and the controller has to absorb the clusters.

In [ ]:
POISSON_STUDY = STUDY.with_(name="arrival_poisson", arrival="poisson")
print(POISSON_STUDY.n_runs, "runs,", f"{POISSON_STUDY.bench_hours:.1f} h")
# nb.experiments.launch(POISSON_STUDY, interval=INTERVAL_S)